
# VeloxQ ERP Scheduling Experiments

This notebook tests the VeloxQ SDK and the existing scheduling QUBOs.

VeloxQ is a physics-inspired remote optimizer, not a physical
quantum processor.

Run remote submission cells individually.


In [1]:
import sys
from pathlib import Path

# Find the repository root
cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path for path in (cwd, *cwd.parents)
        if (path / "quantum_erp" / "model.py").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Cannot find quantum_erp package starting from {cwd}"
    )

# Make local package importable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

# Verify
import quantum_erp.model
import quantum_erp.qubo
import quantum_erp.veloxq_backend

print("All local project modules imported successfully.")

Project root: /mnt/c/Users/kubam/Downloads/quantum_erp_hackathon_uv/quantum_erp_hackathon
All local project modules imported successfully.


In [2]:

import os
import numpy as np
import dimod

from veloxq_sdk import VeloxQSolver

from quantum_erp.model import (
    baseline_a,
    baseline_b,
    milling_outage,
    schedule_records,
    validate,
)

from quantum_erp.qubo import build_qubo
from quantum_erp.reference import exact_reference
from quantum_erp.veloxq_backend import (
    as_spin_bqm,
    verify_spin_encoding,
    sample_velox_qubo,
)

print("VeloxQ token configured:", bool(os.getenv("VELOX_TOKEN")))
print("dimod version:", dimod.__version__)


VeloxQ token configured: True
dimod version: 0.12.22


## Test 1 — Local QUBO-to-Ising verification

In [3]:

cases = [
    (baseline_a(), 4),
    (baseline_b(), 8),
    (milling_outage(), 10),
]

for factory, H in cases:
    qubo = build_qubo(factory, H)
    ising = verify_spin_encoding(qubo)

    print(
        factory.name,
        "H=", H,
        "variables=", len(qubo.variables),
        "couplings=", len(qubo.quadratic),
        "SPIN variables=", len(ising.variables),
    )


instance_2x2 H= 4 variables= 8 couplings= 12 SPIN variables= 8
instance_3x3 H= 8 variables= 39 couplings= 181 SPIN variables= 39
extension_milling_outage H= 10 variables= 50 couplings= 277 SPIN variables= 50


## Test 2 — Official VeloxQ API smoke test

In [4]:

assert os.getenv("VELOX_TOKEN"), "Set VELOX_TOKEN first"

solver = VeloxQSolver()

biases = [1, -1, 0]
couplings = [
    [0, -1, 0],
    [-1, 0, -1],
    [0, -1, 0],
]

smoke = solver.sample(biases, couplings)

print("First sample:", smoke.first)
print("Returned variable type:", smoke.vartype)
print("Number of returned samples:", len(smoke))


First sample: Sample(sample={'0': np.int8(1), '1': np.int8(1), '2': np.int8(1)}, energy=np.float32(-2.0), num_occurrences=np.int64(1825))
Returned variable type: Vartype.SPIN
Number of returned samples: 3


## Test 3 — Original one-bit 2×2 QUBO

In [5]:

# E(x) = 4 + x, with x=0 meaning Z2 then Z1.

binary_bqm = dimod.BinaryQuadraticModel(
    {0: 1.0},
    {},
    4.0,
    dimod.BINARY,
)

spin_bqm = binary_bqm.change_vartype(
    dimod.SPIN,
    inplace=False,
)

response = VeloxQSolver().sample(spin_bqm)

assert response.vartype == dimod.SPIN

sample = response.first.sample

assert len(sample) == 1, (
    f"Expected one variable, received {list(sample.keys())}"
)

spin = int(next(iter(sample.values())))

assert spin in (-1, 1), f"Invalid Ising spin: {spin}"

print("Returned variable labels:", list(sample.keys()))
x = (spin + 1) // 2

print("Returned spin:", spin)
print("Decoded binary variable:", x)
print("Locally recomputed makespan:", 4 + x)
print("Reference optimum:", 4)


Returned variable labels: ['0']
Returned spin: -1
Decoded binary variable: 0
Locally recomputed makespan: 4
Reference optimum: 4


## Test 4 — 2×2 time-indexed scheduling QUBO

In [6]:

factory = baseline_a()
qubo = build_qubo(factory, 4)

result = sample_velox_qubo(qubo)

print("Locally computed energy:", result["energy"])
print("Zero-energy state found:", result["feasible_horizon"])

if result["feasible_horizon"]:
    starts = qubo.starts(result["bits"])
    schedule = schedule_records(factory, starts)
    print("Validated makespan:", validate(factory, schedule))
    print("Schedule:", schedule)
else:
    print("No feasible candidate found; this is not an infeasibility proof.")


Locally computed energy: 0.0
Zero-energy state found: True
Validated makespan: 4
Schedule: [{'job': 'Z1', 'machine': 'T', 'start': 1, 'end': 3}, {'job': 'Z1', 'machine': 'F', 'start': 3, 'end': 4}, {'job': 'Z2', 'machine': 'T', 'start': 0, 'end': 1}, {'job': 'Z2', 'machine': 'F', 'start': 1, 'end': 3}]


## Test 5 — Full 3×3 QUBO

In [7]:

factory = baseline_b()
qubo = build_qubo(factory, 8)

result = sample_velox_qubo(qubo)

print("Local energy:", result["energy"])
print("Samples inspected:", result["samples_inspected"])

if result["feasible_horizon"]:
    starts = qubo.starts(result["bits"])
    schedule = schedule_records(factory, starts)

    makespan = validate(factory, schedule)
    reference = exact_reference(factory)["makespan"]

    print("VeloxQ makespan:", makespan)
    print("Exact classical reference:", reference)
    print("Schedule:", schedule)
else:
    print("No feasible solution found by the remote sampler.")


Local energy: 0.0
Samples inspected: 101
VeloxQ makespan: 8
Exact classical reference: 8
Schedule: [{'job': 'Z1', 'machine': 'T', 'start': 1, 'end': 3}, {'job': 'Z1', 'machine': 'F', 'start': 3, 'end': 4}, {'job': 'Z1', 'machine': 'S', 'start': 5, 'end': 7}, {'job': 'Z2', 'machine': 'T', 'start': 0, 'end': 1}, {'job': 'Z2', 'machine': 'F', 'start': 1, 'end': 3}, {'job': 'Z2', 'machine': 'S', 'start': 4, 'end': 5}, {'job': 'Z3', 'machine': 'T', 'start': 3, 'end': 5}, {'job': 'Z3', 'machine': 'F', 'start': 5, 'end': 7}, {'job': 'Z3', 'machine': 'S', 'start': 7, 'end': 8}]


## Test 6 — Stage 2 machine-outage QUBO

In [8]:

factory = milling_outage()
qubo = build_qubo(factory, 10)

result = sample_velox_qubo(qubo)

print("Local energy:", result["energy"])
print("Samples inspected:", result["samples_inspected"])

if result["feasible_horizon"]:
    starts = qubo.starts(result["bits"])
    schedule = schedule_records(factory, starts)

    makespan = validate(factory, schedule)
    reference = exact_reference(factory)["makespan"]

    print("VeloxQ makespan:", makespan)
    print("Exact classical reference:", reference)
    print("Schedule:", schedule)
else:
    print("No feasible solution found by the remote sampler.")


Local energy: 0.0
Samples inspected: 296
VeloxQ makespan: 10
Exact classical reference: 10
Schedule: [{'job': 'Z1', 'machine': 'T', 'start': 4, 'end': 6}, {'job': 'Z1', 'machine': 'F', 'start': 6, 'end': 7}, {'job': 'Z1', 'machine': 'S', 'start': 7, 'end': 9}, {'job': 'Z2', 'machine': 'T', 'start': 6, 'end': 7}, {'job': 'Z2', 'machine': 'F', 'start': 7, 'end': 9}, {'job': 'Z2', 'machine': 'S', 'start': 9, 'end': 10}, {'job': 'Z3', 'machine': 'T', 'start': 1, 'end': 3}, {'job': 'Z3', 'machine': 'F', 'start': 4, 'end': 6}, {'job': 'Z3', 'machine': 'S', 'start': 6, 'end': 7}]
